# P11–P15 state, governance, and harness evidence

## Scope and authority

This notebook displays later-paper state, robustness, responsibility, RSE, and evidence-admission receipts. Internal authorship, finite corruption worlds, and cryptographic validity are kept distinct from independent scientific authority.


## Theory, methodology, and algorithms

A support count can be viewed schematically as

$$s_f(\tau)=\sum_{i=1}^{n}\mathbf{1}\{d(f(x_i),y_i)\le\tau\},$$

while forward-time evaluation requires a temporal split

$$\max(t_{\mathrm{train}})<\min(t_{\mathrm{test}}).$$

For responsibility experiments, a finite matrix indexes arm $a$ and corruption world $w$: $R_{a,w}$. For P15, the central non-implication is

$$\operatorname{Verify}(pk,m,\sigma)=1\;\not\Rightarrow\;\operatorname{True}(m).$$

A signature authenticates a statement under a key; it does not establish key custody, fact truth, scientific validity, or claim authority.


In [ ]:
from pathlib import Path
import json
import sys


def find_visualization_root(start=Path.cwd()):
    """Find visualization/ whether Jupyter starts at the repo root or notebooks/."""
    start = start.resolve()
    candidates = [start / "visualization", start, *start.parents]
    for candidate in candidates:
        if candidate.name == "visualization" and (candidate / "data" / "derived" / "atlas.json").exists():
            return candidate
        nested = candidate / "visualization"
        if (nested / "data" / "derived" / "atlas.json").exists():
            return nested
    raise FileNotFoundError(
        "Could not find visualization/data/derived/atlas.json. "
        "Build the atlas from the repository root first."
    )


VIS_ROOT = find_visualization_root()
sys.path.insert(0, str(VIS_ROOT / "src"))
ATLAS_PATH = VIS_ROOT / "data" / "derived" / "atlas.json"
atlas = json.loads(ATLAS_PATH.read_text(encoding="utf-8"))


def as_rows(value):
    """Return normalized records without changing their scientific values."""
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        return [row for row in value.values() if isinstance(row, dict)]
    return []


def first(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def paper_id(row):
    raw = str(first(row, "paper_id", "paper", "id", default="UNSCOPED"))
    return raw.replace("ORION-", "")


def exact_status(row):
    return str(first(row, "terminal", "status", "result_state", "authority", default="UNSPECIFIED"))


def numeric_value(row):
    value = first(row, "value", "observed", "count", default=None)
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


paper_states = as_rows(atlas.get("paper_states", []))
metrics_by_paper = atlas.get("metrics", {})
metrics = as_rows(atlas.get("metric_records", []))
anomalies = as_rows(atlas.get("anomalies", []))
sources = as_rows(atlas.get("sources", []))
des_execution = as_rows(atlas.get("des_execution", []))
framework_mechanics = atlas.get("framework_mechanics", {})

print(f"Atlas: {ATLAS_PATH}")
print(
    f"Loaded {len(paper_states)} paper states, {len(metrics)} metrics, "
    f"{len(anomalies)} anomalies, {len(des_execution)} frozen DES rows and "
    f"{len(sources)} sources."
)


In [ ]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import ListedColormap  # noqa: F401 -- used by heatmap notebooks

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

STATE_COLORS = {
    "PASS": "#2e7d32",
    "SUPPORTED": "#2e7d32",
    "FAIL": "#c62828",
    "GATE_NOT_MET": "#c62828",
    "CANNOT_CHECK": "#ef6c00",
    "UNKNOWN": "#6a1b9a",
    "NOT_AUTHORITY": "#455a64",
    "NOT_EXECUTED": "#757575",
}


def state_color(text):
    upper = str(text).upper()
    for token, color in STATE_COLORS.items():
        if token in upper:
            return color
    return "#1565c0"


def human_label(value, width=18):
    # Wrap machine identifiers without changing canonical capitalization.
    cleaned = str(value).replace("_", " ").replace(":", " — ")
    return "\n".join(textwrap.wrap(cleaned, width=width, break_long_words=False))



def print_records(rows, fields, limit=30):
    """Small dependency-free table for exact atlas fields."""
    rows = list(rows)
    if not rows:
        print("No records match the current display selectors.")
        return
    widths = {
        field: min(
            48,
            max(len(field), *(len(str(first(row, field, default=""))) for row in rows[:limit])),
        )
        for field in fields
    }
    print(" | ".join(field.ljust(widths[field]) for field in fields))
    print("-+-".join("-" * widths[field] for field in fields))
    for row in rows[:limit]:
        print(" | ".join(str(first(row, field, default=""))[: widths[field]].ljust(widths[field]) for field in fields))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more record(s); change DISPLAY_LIMIT to inspect them.")


## Editable selectors and thresholds

Select one exact numeric metric for a raw-value bar plot. The display threshold does not alter terminals and should not be used to hide adverse rows.


In [ ]:
PAPERS = ["P11", "P12", "P13", "P14", "P15"]
available_metric_names = sorted({str(first(row, "metric", "name", "metric_name", default="")) for row in metrics if paper_id(row) in PAPERS and numeric_value(row) is not None})
METRIC_NAME = "unsafe_reuse_rate" if "unsafe_reuse_rate" in available_metric_names else (available_metric_names[0] if available_metric_names else None)
MIN_VALUE = None
DISPLAY_LIMIT = 60
print("Available exact metric names:", available_metric_names)
print("Selected:", METRIC_NAME)


## Results: selected exact metric

Horizontal bars show individual normalized atlas rows on their natural scale. Repeated paper labels mean the receipt supplies multiple rows; the notebook does not average them. For the default unsafe-reuse view, lower is better and rate values use the full 0–1 domain.


In [ ]:
selected_metrics = [
    row for row in metrics
    if paper_id(row) in PAPERS
    and numeric_value(row) is not None
    and (METRIC_NAME is None or str(first(row, "metric", "name", "metric_name", default="")) == METRIC_NAME)
    and (MIN_VALUE is None or numeric_value(row) >= MIN_VALUE)
]
fig, ax = plt.subplots(figsize=(10.5, 5.5))
if selected_metrics:
    rows = sorted(selected_metrics, key=lambda row: numeric_value(row), reverse=True)
    display_names = {
        "UNQUALIFIED": "Unqualified",
        "CONFIDENCE_ONLY": "Confidence only",
        "UNVERIFIED_RCS": "Unverified RCS",
        "ALWAYS_RAW": "Always raw",
        "AUTHENTICATED_RCS": "Authenticated RCS",
    }
    labels = [
        f"{paper_id(row)} — {human_label(display_names.get(str(first(row, 'case_id', 'arm', 'label', 'name', default=METRIC_NAME)), first(row, 'case_id', 'arm', 'label', 'name', default=METRIC_NAME)), 28)}"
        for row in rows
    ]
    values = [numeric_value(row) for row in rows]
    unit = str(first(rows[0], "unit", default="receipt unit"))
    metric_label = human_label(METRIC_NAME, 40)
    is_unsafe_reuse = METRIC_NAME == "unsafe_reuse_rate"
    direction_note = "; lower is better" if is_unsafe_reuse else ""
    y = list(range(len(values)))
    bars = ax.barh(y, values, height=0.62, color="#455a64")
    ax.set_yticks(y, labels)
    ax.invert_yaxis()
    ax.set_xlabel(f"{metric_label} ({unit}{direction_note})")
    ax.set_ylabel("paper and receipt row")
    ax.set_title(
        "Unsafe reuse by P13 control arm (bounded internal receipts)"
        if is_unsafe_reuse
        else f"Raw receipt values: {metric_label}"
    )
    if unit.lower() in {"rate", "fraction", "proportion", "probability"}:
        ax.set_xlim(0, 1)
    for bar, value in zip(bars, values):
        ax.annotate(
            f"{value:.1%}" if unit.lower() == "rate" else f"{value:g}",
            (value, bar.get_y() + bar.get_height() / 2),
            xytext=(6, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
        )
    fig.text(
        0.5,
        0.01,
        (
            "A zero bar is a bounded receipt value, not evidence of external safety authority."
            if is_unsafe_reuse
            else "Receipt-level display only; favorable direction and external authority are not inferred."
        ),
        ha="center",
        fontsize=9,
        color="#455a64",
    )
else:
    ax.text(0.5, 0.5, "No matching numeric records", ha="center", va="center")
    ax.set_axis_off()
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()


In [ ]:
governance_states = [row for row in paper_states if paper_id(row) in PAPERS]
governance_anomalies = [row for row in anomalies if paper_id(row) in PAPERS]
print("EXACT STATES")
print_records(governance_states, ["paper_id", "title", "status", "terminal", "authority", "claim_ceiling"], DISPLAY_LIMIT)
print("\nSELECTED METRIC ROWS")
print_records(selected_metrics, ["paper_id", "metric", "name", "value", "unit", "case_id", "arm", "status"], DISPLAY_LIMIT)
print("\nANOMALIES")
print_records(governance_anomalies, ["paper_id", "anomaly_id", "severity", "status", "summary"], DISPLAY_LIMIT)


## Discussion: gate and authority boundaries

- **P11:** 30 query results and support counts `LINEAR=3`, `RBF=5`, `KNN=5` coexist with terminal `GATE_NOT_MET`; counts do not override the gate.
- **P12:** 32 receipt families span $\sigma\in\{0.2,0.4,0.6,0.8\}$; the registered active-authority receipt says `forward_time_deployability=CANNOT_CHECK` and `campaign_executed=false` for the public-data stop/go campaign.
- **P13:** the five-arm/four-corruption-world result is bounded finite. A separately registered P13A receipt retains historical terminal `P14_CONTROLLED_SUFFICIENCY_DEBT_GATE_NOT_MET` with observed maximum deviation $0.0556640625>0.05$; the exact historical label is preserved rather than silently renamed.
- **P14:** the 28 internally authored cases are not an external pilot. The separate 67-packet pilot analytics receipt is `NOT_AUTHORITY` because frontier-agent execution and independent human adjudication have not run.
- **P15:** four real-workflow receipts include three `AUTHORIZED_SCIENCE` receipt dispositions and one honest `CANNOT_CHECK`; these are not publication/external-authority labels. Separately, the active-authority receipt records 0 signature-layer detections and 6 false promotions under full key compromise, exposing the boundary: valid signatures do not prove custody or truth.

## Claim ceiling

The displayed receipts support only their registered finite/local statements. They do not establish forward-time transfer, external-pilot independence, production-scale reliability, secure key custody, or scientific truth from provenance/signatures.
